# Attention-Aware Attribution

Upstream circuit-tracer builds its attribution graphs by the circuit-tracing method of Ameisen et al.
<a id="cite-1"></a>[[1]](#ref-1), which holds the attention patterns and the norm denominators constant. That is what makes the
graph's edges exact direct effects in a linearized model, and it is also a blind spot: an effect that acts by changing
*where attention looks* is credited to no node, and an edge does not say which head carried it.

This notebook demonstrates three attention-aware additions to the analysis surface, each through its top-level
wrapper, on one prompt whose answer is easy to state:

- **The two regimes:** `compute_attribution_graph` with patterns frozen (the default) and with patterns live
  (`freeze_attention=False`). A live-pattern edge is a first-order derivative, not an exact direct effect.
- **Head loadings:** `it.head_loadings(...)` splits the edges into one target across the heads of a layer.
- **Head conductance:** `it.head_conductance(...)` credits each head for the measured change of a target
  between two prompts of the same length, split into a pattern (QK) part and a value (OV) part.

**Where the additions live.** All three are implemented in Interpretune's fork of circuit-tracer
(`speediedan/circuit-tracer`), which Interpretune pins and installs; Interpretune exposes them as the op wrappers used
below. They are not in upstream circuit-tracer <a id="cite-2a"></a>[[2]](#ref-2). Offering them upstream is tracked in
[#980](https://github.com/speediedan/interpretune/issues/980), and how Interpretune ships the fork until then in [#1018](https://github.com/speediedan/interpretune/issues/1018). Two upstream proposals by
other contributors overlap them: opt-in attention and layer-norm freezing flags
([decoderesearch/circuit-tracer#123](https://github.com/decoderesearch/circuit-tracer/pull/123)) and head loadings for the TransformerLens backend
([decoderesearch/circuit-tracer#114](https://github.com/decoderesearch/circuit-tracer/pull/114)).

Alongside each report, the [analysis injection](../attribution_analysis/attribution_analysis.ipynb) framework
reads the intermediate values circuit-tracer builds it from: which locations each regime detaches, the
Neumann series behind node influence, the per-head copies behind a head loading, and the path a conductance
integral follows.

The nnsight model-level adapter is the first to support these reports; support in other model-level adapters
is planned.

**What to expect.** Each section measures something the frozen graph cannot show, and the closing section brings them
together: where the two regimes disagree (mostly on intermediate edges that cross positions), how that moves pruning,
which heads carry a measured change, how the change divides between attention and MLP outputs, what editing each
one does to the answer, and where the unstated *state* changes from Texas to Michigan.

In [ ]:
# @title Imports { display-mode: "form" }
import importlib.util
import json
import math
import tempfile
from pathlib import Path
from pprint import pformat

import numpy as np
import torch

import interpretune as it
import interpretune.analysis  # registers the top-level op wrappers used below
from it_examples.utils.attention_attribution_helpers import (
    INTERVENTION_ARMS,
    align_regime_nodes,
    apply_last_token_edits,
    capture_resid_post,
    capture_residual_terms,
    check_next_token,
    conductance_path,
    display_next_token_checks,
    display_plotly_figure,
    influence_by_kind,
    intervention_arm_sets,
    last_token_edits,
    logit_lens_contrast,
    module_tokenizer,
    node_kinds,
    node_positions,
    partition_final_residual,
    plot_conductance_path,
    plot_dose_response,
    plot_head_heatmap,
    plot_head_split,
    plot_layer_bars,
    plot_neumann_convergence,
    plot_position_change_mass,
    plot_regime_edges,
    plot_residual_convergence,
    plot_running_credit,
    plot_state_read,
    position_change_mass,
    resolve_feature_context,
    resolve_intervention_arms,
    top_k_overlap,
)
from it_examples.utils.example_helpers import required_os_env
from it_examples.utils.nb_ui_utils import display_notebook_table, display_topk_token_predictions
from interpretune import ITSession, ITSessionConfig
from interpretune.config import AnalysisCfg, init_analysis_cfgs

## Notebook Parameters

This cell contains parameters that can be injected by papermill during parameterized test runs.
`conductance_steps` is the step count of the path integral in the conductance section: a path that moves
sharply needs many more steps than the default, which is exactly what its completeness residual reports.
`residual_sweep_steps` are the step counts the convergence check compares. `influence_panel` lists the
(city, capital) prompts the node-influence comparison is repeated on, and `intervention_doses` the doses each
intervention arm is applied at. `second_model` is the model of another family the intervention arms are run on as
well, through the same addresses.

In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)
conductance_steps = 32  # Path-integral steps for head_conductance; read its residual before trusting few steps
head_layers = None  # Layers for the head-loadings report (None reports every layer)
residual_sweep_steps = [8, 16, 32, 64, 128]  # Step counts for the completeness-residual convergence check
influence_panel = [  # (city, capital) prompts for the paired node-influence comparison; one state each
    ("Dallas", "Austin"),
    ("Detroit", "Lansing"),
    ("Chicago", "Springfield"),
    ("Miami", "Tallahassee"),
    ("Seattle", "Olympia"),
    ("Cleveland", "Columbus"),
    ("Philadelphia", "Harrisburg"),
    ("Milwaukee", "Madison"),
    ("Portland", "Salem"),
    ("Buffalo", "Albany"),
]
intervention_doses = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]  # multiples of the Dallas-to-Detroit change
enable_analysis_injection = True  # Read circuit-tracer's intermediate values through the analysis points
analysis_config_path = "attention_attribution_injection_config.yaml"  # Analysis injection config beside this notebook
neuronpedia_model_id = "gemma-2-2b"  # feature descriptions and links come from public Neuronpedia
neuronpedia_source_set = "gemmascope-transcoder-16k"
second_model = "meta-llama/Llama-3.2-1B"  # another family the intervention arms run on unchanged (None skips it)

In [ ]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to '/full/path/to/.env' to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

## Analysis Injection

The analysis points are patched into circuit-tracer before anything imports it, so they are set up first. Each
point records into a trace that the cells below read and reset; the points and what each reads are in
`attention_attribution_points.py`, and the lines they attach to are in `attention_attribution_injection_config.yaml`.
The [attribution analysis](../attribution_analysis/attribution_analysis.ipynb) notebook walks through the
framework itself.

In [ ]:
NOTEBOOK_DIR = Path.cwd()
config_path = Path(analysis_config_path)
config_path = (config_path if config_path.is_absolute() else NOTEBOOK_DIR / config_path).resolve()
log_dir = Path(core_log_dir).expanduser() if core_log_dir else Path(tempfile.gettempdir())
log_dir.mkdir(parents=True, exist_ok=True)

# the notebook reads the traces through the same module the orchestrator loads the points from
_points_spec = importlib.util.spec_from_file_location(
    "attention_attribution_points", config_path.parent / "attention_attribution_points.py"
)
points = importlib.util.module_from_spec(_points_spec)
_points_spec.loader.exec_module(points)

analysis_injector = None
if enable_analysis_injection:
    from interpretune.analysis.injection import orchestrator

    analysis_injector = orchestrator.setup_analysis_injection(
        config_path=config_path,
        target_package="circuit_tracer",
        config_overrides=f"settings:\n  log_dir: {log_dir.as_posix()}\n",
    )
    print(f"Analysis points active: {sorted(analysis_injector.config.file_hooks)}")
points.reset_trace()

## Session Configuration

We load the model-only configuration of the task-neutral `speediedan/prompt-analysis` component for Gemma-2-2b via
`it.hub.load`, with the nnsight backend and the Gemma Scope transcoder set, the same substrate as the
[CT analysis backend demo](ct_analysis_backend_demo.ipynb). It carries no prompts and no attribution targets: the
prompts and targets below are this notebook's own.

In [ ]:
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load(
    "speediedan/prompt-analysis", "prompt_analysis.gemma2.circuit_tracer"
)

base_it_cfg.circuit_tracer_cfg.backend = "nnsight"
base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"
base_it_cfg.circuit_tracer_cfg.verbose = False
if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir
print(pformat(base_it_cfg.circuit_tracer_cfg))

session_cfg = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)

module = it_session.module
model = module.replacement_model
tokenizer = model.tokenizer
module.analysis_cfg = AnalysisCfg(target_op=it.compute_attribution_graph, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])
print(f"Module type: {type(module).__name__}")
print("Session initialized successfully!")

## One Prompt, One Swap, and a Sanity Check

The prompt is the capital prompt from circuit-tracer's tutorial and the backend demo: the answer, Austin,
needs one unstated step (Dallas is in Texas). In this notebook *state* in italics means one of the
United States (Texas, Michigan), not a condition of the model. It is kept so the graphs here stay comparable with upstream's.

The counterfactual swaps Dallas for **Detroit**, which `head_conductance` needs to be the same token length:
it integrates along the line between the two prompts' embeddings, so they must align position by position.
The swapped city is a large city that is *not* its *state*'s capital, so the answer to the swapped prompt is a
different capital (Lansing) reached by the same unstated step. A capital city would confound the comparison:
for "the state containing Denver" the right answer is Denver itself, so copying the stated city and retrieving
a capital give the same token, and the measured change could not separate them.

The check below states what the comparison depends on before anything is attributed: on each prompt the
model's top answer is the expected capital, with a clear margin over the next token, and copying the stated
city is far less likely than answering.

In [ ]:
prompt = "Fact: the capital of the state containing Dallas is"
swapped_prompt = "Fact: the capital of the state containing Detroit is"
answer, swapped_answer = "▁Austin", "▁Lansing"

prompt_ids = tokenizer.encode(prompt)
swapped_ids = tokenizer.encode(swapped_prompt)
assert len(prompt_ids) == len(swapped_ids), "head_conductance needs two prompts of the same token length"
changed = [i for i, (a, b) in enumerate(zip(prompt_ids, swapped_ids, strict=True)) if a != b]
assert len(changed) == 1, f"the swap should change exactly one position, changed {changed}"
prompt_tokens = [tokenizer.decode([t]) for t in prompt_ids]
city_pos = changed[0]
print(f"Prompt tokens: {list(enumerate(prompt_tokens))}")
print(f"Changed position: {city_pos} ({prompt_tokens[city_pos]!r} -> {tokenizer.decode([swapped_ids[city_pos]])!r})")

checks = [
    check_next_token(model, tokenizer, "Dallas", prompt, answer, copy_token="▁Dallas"),
    check_next_token(model, tokenizer, "Detroit", swapped_prompt, swapped_answer, copy_token="▁Detroit"),
]
display_next_token_checks(checks)
for c in checks:
    assert c.rank == 1, f"{c.label}: the model must answer {c.expected!r} first, got {c.top[:3]}"
    assert c.margin >= 1.0, f"{c.label}: {c.expected!r} leads by only {c.margin:.2f} logits"
    assert c.copy_prob < 0.25 * c.prob, f"{c.label}: copying the stated city is too likely ({c.copy_prob:.3f})"

austin_id = tokenizer.encode(answer, add_special_tokens=False)[-1]
lansing_id = tokenizer.encode(swapped_answer, add_special_tokens=False)[-1]
dallas_id = tokenizer.encode("▁Dallas", add_special_tokens=False)[-1]
state_ids = {w: tokenizer.encode(w, add_special_tokens=False) for w in ("▁Texas", "▁Michigan", "▁Detroit")}
assert all(len(ids) == 1 for ids in state_ids.values()), f"each must be one token: {state_ids}"
texas_id, michigan_id, detroit_id = (ids[0] for ids in state_ids.values())


def feature_context(features):
    """Token text, a Neuronpedia description and a link for the (layer, pos, feature) triples a table shows."""
    return resolve_feature_context(
        features, prompt_tokens, model_id=neuronpedia_model_id, source_set=neuronpedia_source_set
    )

## The Two Regimes Disagreeing

The same graph computed twice: once with the default regime (patterns and norms frozen) and once with
patterns live. The regime is recorded on each graph, and the record matters: a graph whose record was lost
reads as frozen, which would mislabel a patterns-live graph with no error anywhere.

Under the live regime an edge is a first-order derivative of the target with respect to the source, read at
this prompt. It credits a source for changing where attention looks, at the price of exactness: the frozen
graph's edges are exact direct effects in the linearized model, and the live graph's are not.

The `ap_gradient_flow_regime` analysis point shows the mechanism: before the backward pass, circuit-tracer
detaches the attention-pattern outputs in the frozen regime and leaves them attached in the live one.

**The graphs name their targets.** The configuration loaded above names none, so circuit-tracer would fall back to
its salient-logit selection; named targets replace that selection, so the graphs here name theirs: Austin and
Lansing, the two answers the comparison is about, and Dallas, the token copying the city would produce. Lansing is improbable on this prompt and is a target anyway, so the graph has an Austin row and a
Lansing row to subtract.

**Why the norm denominators stay frozen in both graphs.** The live graph frees the attention patterns only; its
record below reads `freeze_norms: true`. When an RMSNorm's denominator is live, a change to its input also rescales its
whole output. That rescaling is shared by every source writing to the position, so it belongs to no node, and a graph
would spread it over all of that position's edges. Measured on a mid-layer target, letting the denominators move
cancelled most of the predicted effect ([#660](https://github.com/speediedan/interpretune/issues/660)), and representing them as graph nodes of their own is open
work ([#684](https://github.com/speediedan/interpretune/issues/684)). A logit target is less exposed, because its attribution concentrates in late layers. The fork
accepts `freeze_norms=False` for anyone who wants to measure the difference.

**Compare the two graphs node by node, not index by index.** Each graph selects its own feature nodes from the
prompt's active features, by attribution, so the two adjacency matrices have the same shape while their feature
columns name different features. `align_regime_nodes` maps both onto the features they both selected, plus the
error, token and logit nodes, which the two lay out identically; every comparison below reads those shared nodes.

In [ ]:
graph_targets = torch.tensor([austin_id, lansing_id, dallas_id])
points.reset_trace(points.GRADIENT_FLOW)
frozen = it.compute_attribution_graph(
    module, None, None, 0, prompt=prompt, attribution_targets=graph_targets, verbose=False
)
live = it.compute_attribution_graph(
    module,
    None,
    None,
    0,
    prompt=prompt,
    attribution_targets=graph_targets,
    verbose=False,
    freeze_attention=False,
)

flows = (
    {flow["freeze_attention"]: flow for flow in points.read_trace(points.GRADIENT_FLOW)}
    if enable_analysis_injection
    else {}
)
rows = []
for name, graph in (("frozen", frozen), ("live", live)):
    regime = json.loads(graph.graph_attribution_regime_json)
    flow = flows.get(regime.get("freeze_attention"))
    detached = (
        f"{flow['detached']} of {flow['attention_locations'] + flow['norm_scale_locations']} "
        f"({flow['attention_locations']} attention-pattern and {flow['norm_scale_locations']} norm-scale locations)"
        if flow
        else "not traced"
    )
    rows.append([name, json.dumps(regime), detached])
display_notebook_table(
    rows,
    ["Graph", "Recorded regime", "Locations detached before the backward pass"],
    title="What each regime holds constant",
    caption="Read from the graph's own record and, beside it, from the ap_gradient_flow_regime analysis point.",
)
if enable_analysis_injection:
    assert set(flows) == {True, False}, f"the gradient-flow point must see both regimes, saw {set(flows)}"
target_probs = [round(float(p), 4) for p in frozen.logit_probabilities]
print(f"Logit targets: {frozen.logit_target_tokens} (probabilities on this prompt {target_probs})")

aligned = align_regime_nodes(frozen, live)
same_index = int((frozen.selected_features == live.selected_features).sum())
frozen_adj = frozen.adjacency_matrix.index_select(0, aligned.first).index_select(1, aligned.first).float()
live_adj = live.adjacency_matrix.index_select(0, aligned.second).index_select(1, aligned.second).float()
delta = live_adj - frozen_adj
display_notebook_table(
    [
        ["Feature nodes selected", len(frozen.selected_features), len(live.selected_features)],
        ["Feature nodes both graphs selected (matched by identity)", aligned.n_common, aligned.n_common],
        ["Of those, at the same index in both graphs", same_index, same_index],
    ],
    ["Count of feature nodes", "frozen", "live"],
    title="Putting the two graphs on the same nodes",
    caption="Every entry is a count of nodes, not a feature id.",
)
n_changed, n_entries = int((delta != 0).sum()), delta.numel()
print(
    f"Edges between the shared nodes that differ between the regimes: {n_changed:,} of {n_entries:,} adjacency "
    f"entries ({n_changed / n_entries:.1%}). The entry count includes zeros: most pairs of nodes have no edge in "
    "either regime."
)
assert float(delta.abs().max()) > 0, "positive control: the two regimes must actually disagree"

### Where the regimes disagree

Rows of the adjacency matrix are target nodes and columns are sources, in the order features, error nodes, token
(embed) nodes, logit nodes. Before reading any one row, the table below asks where the change between the regimes
lives. Almost all of it is on edges INTO features, not into the logits: freezing the patterns changes how the graph
credits its intermediate steps far more than how it credits the answer directly, so a comparison that reads only the
Austin row reads a sliver of the disagreement.

The heatmap groups the feature-to-feature changes by position. A feature can only reach a feature at another
position through attention, so mass off the diagonal is credit that moves between positions, and it is exactly the
credit the frozen regime handles differently: through values alone, never through where attention looks.

In [ ]:
n_pos = len(frozen.input_tokens)
n_layers = model.cfg.n_layers
target_ids = [int(t) for t in frozen.logit_target_ids]
n_logits = len(target_ids)
n_common = aligned.n_common
n_aligned = len(aligned.first)
assert n_common + (n_layers + 1) * n_pos + n_logits == n_aligned, "unexpected node layout"
kinds = np.asarray(node_kinds(n_common, n_layers, n_pos, n_logits))
positions = node_positions(aligned.common_features, n_layers, n_pos, n_logits)
logit_names = [t.strip() for t in frozen.logit_target_tokens]


def aligned_node(j):
    """(kind, layer, pos, feature) of the j-th aligned node; layer, pos or feature is None where it does not apply."""
    if j < n_common:
        layer, pos, feat = (int(v) for v in aligned.common_features[j])
        return "feature", layer, pos, feat
    k = j - n_common
    if k < n_layers * n_pos:
        return "error", k // n_pos, k % n_pos, None
    k -= n_layers * n_pos
    if k < n_pos:
        return "token", None, k, None
    return "logit", None, None, k - n_pos


def aligned_label(j):
    kind, layer, pos, feat = aligned_node(j)
    if kind == "feature":
        return f"L{layer} #{feat} on {prompt_tokens[pos]!r} (pos {pos})"
    if kind == "error":
        return f"error L{layer} on {prompt_tokens[pos]!r} (pos {pos})"
    if kind == "token":
        return f"token {prompt_tokens[pos]!r} (pos {pos})"
    return f"logit {logit_names[feat]}"


total_change = float(delta.abs().sum())
row_groups = [("into features", torch.as_tensor(np.flatnonzero(kinds == "feature")))]
row_groups += [
    (f"into the {name} logit", torch.tensor([n_aligned - n_logits + i])) for i, name in enumerate(logit_names)
]
rows = []
for label, idx in row_groups:
    block_change, block_frozen = float(delta[idx].abs().sum()), float(frozen_adj[idx].abs().sum())
    rows.append([label, f"{block_change:,.1f}", f"{block_change / total_change:.2%}", f"{block_frozen:,.1f}"])
display_notebook_table(
    rows,
    ["Edges", "Summed |live - frozen|", "Share of all change", "Summed |frozen|"],
    title="Where the change between the regimes lives",
    caption="Error and token nodes have no incoming edges, so the two row groups cover every edge.",
)

pos_mass = position_change_mass(delta, positions, kinds, n_pos)
cross = float(pos_mass.sum() - np.trace(pos_mass)) / float(pos_mass.sum())
print(f"Feature-to-feature change that crosses positions: {cross:.1%}")
display_plotly_figure(
    plot_position_change_mass(
        pos_mass, prompt_tokens, "Feature-to-feature change by position: target row, source column"
    ),
    "it-position-change",
)

feature_idx = torch.as_tensor(np.flatnonzero(kinds == "feature"))
block = delta.index_select(0, feature_idx).index_select(1, feature_idx).abs()
top_edges = [divmod(int(i), len(feature_idx)) for i in block.flatten().topk(8).indices]
shown = [aligned_node(int(feature_idx[r]))[1:] for r, _ in top_edges] + [
    aligned_node(int(feature_idx[c]))[1:] for _, c in top_edges
]
ctx = feature_context(shown)
rows = []
for r, c in top_edges:
    _, tl, tp, tf = aligned_node(int(feature_idx[r]))
    _, sl, sp, sf = aligned_node(int(feature_idx[c]))
    rows.append(
        [
            ctx.link(tl, tf),
            ctx.token(tp),
            ctx.explanation(tl, tf),
            ctx.link(sl, sf),
            ctx.token(sp),
            ctx.explanation(sl, sf),
            f"{float(frozen_adj[feature_idx[r], feature_idx[c]]):+.2f}",
            f"{float(live_adj[feature_idx[r], feature_idx[c]]):+.2f}",
        ]
    )
display_notebook_table(
    rows,
    ["Target", "on", "Target description", "Source", "on", "Source description", "frozen", "live"],
    title="The feature-to-feature edges the regimes disagree on most",
    caption=(
        "Descriptions are Neuronpedia autointerpretations (oai_token-act-pair where one exists): hypotheses about what "
        "a feature responds to, not measured causal roles. A dash means none was available."
    ),
)

### The answer contrast

The question the notebook asks is Austin against Lansing, so the row to read is one contrast target, 'Austin minus
Lansing': for each source, its edge into the Austin logit minus its edge into the Lansing logit. The scatter plots every source's contrast edge,
frozen against live; points on the diagonal are sources the patterns do not change, the labelled points change most,
and hovering a feature shows its description. The table lists the same sources with their descriptions and links.

In [ ]:
a_row = n_aligned - n_logits + target_ids.index(austin_id)
l_row = n_aligned - n_logits + target_ids.index(lansing_id)
frozen_contrast = frozen_adj[a_row] - frozen_adj[l_row]
live_contrast = live_adj[a_row] - live_adj[l_row]
order = (live_contrast - frozen_contrast).abs().topk(8).indices.tolist()
ctx = feature_context([aligned_node(j)[1:] for j in order if aligned_node(j)[0] == "feature"])
labels = [aligned_label(j) for j in range(n_aligned)]
hover = list(labels)
for j in order:
    kind, layer, pos, feat = aligned_node(j)
    if kind == "feature" and ctx.explanation(layer, feat) != "-":
        hover[j] = f"{labels[j]}: {ctx.explanation(layer, feat)}"
display_plotly_figure(
    plot_regime_edges(frozen_contrast, live_contrast, list(kinds), labels, "Austin minus Lansing", hover=hover),
    "it-regime-contrast",
)
rows = []
for j in order:
    kind, layer, pos, feat = aligned_node(j)
    source = ctx.link(layer, feat) if kind == "feature" else aligned_label(j)
    description = ctx.explanation(layer, feat) if kind == "feature" else "-"
    rows.append(
        [
            source,
            kind,
            prompt_tokens[pos] if pos is not None else "-",
            description,
            f"{float(frozen_contrast[j]):+.3f}",
            f"{float(live_contrast[j]):+.3f}",
        ]
    )
display_notebook_table(
    rows,
    ["Source", "Kind", "Token", "Description", "frozen", "live"],
    title="Sources of the 'Austin minus Lansing' contrast that the patterns change most",
    caption=(
        "Edge units are the logit difference's; a live edge is a first-order derivative, a frozen one an exact "
        "direct effect."
    ),
)

### What the change does to node influence

Pruning ranks nodes by influence: the total weight of every path from a node to the logits, through the
row-normalized absolute adjacency $A$. circuit-tracer sums that path series term by term,

$$
w^{\mathsf{T}} (A + A^2 + A^3 + \cdots),
$$

with $w$ the logit probabilities, until a term is zero. The graph is acyclic, so every path has a longest length and
the series ends exactly: its last term is zero, and the plot below stops at the last nonzero term. The `ap_node_influence_*` points record every term, so the
two regimes' series can be compared directly, as the attribution-analysis notebook does for one graph.

**Error nodes and a sum, not a distribution.** An error node holds what a transcoder fails to reconstruct, and its
forward value is the same in both regimes: freezing the patterns changes the backward pass, not the reconstruction. So
the question is not whether the error shrinks but how much of the graph's influence the error nodes take, which is a
sum over nodes. The table reports each node kind's summed influence. Within one prompt the two regimes share the same
logit weights $w$, so their sums compare directly; between prompts they do not (influence is linear in $w$), so the
panel below compares each prompt's two regimes with each other.

In [ ]:
# imported here, after the analysis points are patched in, so the name binds to the patched module
from circuit_tracer.graph import compute_node_influence


def logit_weights(batch):
    weights = torch.zeros(batch.adjacency_matrix.shape[0])
    weights[-len(batch.logit_target_ids) :] = torch.as_tensor(batch.logit_probabilities, dtype=torch.float32)
    return weights


def graph_kinds(batch):
    return node_kinds(len(batch.selected_features), n_layers, len(batch.input_tokens), len(batch.logit_target_ids))


influence, neumann, by_kind = {}, {}, {}
for name, batch in (("frozen", frozen), ("live", live)):
    points.reset_trace(points.NEUMANN_RUNS)
    influence[name] = compute_node_influence(batch.adjacency_matrix.float().cpu(), logit_weights(batch))
    by_kind[name] = influence_by_kind(influence[name], graph_kinds(batch))
    if enable_analysis_injection:
        runs = points.read_trace(points.NEUMANN_RUNS)
        assert len(runs) == 1 and len(runs[0]) > 1, "the Neumann points must record one multi-term series per call"
        neumann[name] = runs[0]
        # positive control: the recorded terms add up to the influence circuit-tracer returned
        assert torch.allclose(torch.stack(runs[0]).sum(0), influence[name], rtol=1e-4, atol=1e-6)

display_notebook_table(
    [
        [
            name,
            f"{v['feature']:.4f}",
            f"{v['error']:.4f}",
            f"{v['token']:.4f}",
            f"{v['total']:.4f}",
            f"{v['error_share']:.2%}",
            len(neumann.get(name, [])) or "not traced",
        ]
        for name, v in by_kind.items()
    ],
    ["Regime", "Features", "Error nodes", "Tokens", "All nodes", "Error share", "Neumann terms"],
    title="Summed node influence by node kind",
    caption=(
        f"Logit weights sum to {float(logit_weights(frozen).sum()):.4f} in both regimes (the targets' probabilities)."
    ),
)
print(
    f"Live minus frozen: error influence {by_kind['live']['error'] - by_kind['frozen']['error']:+.4f}, "
    f"token influence {by_kind['live']['token'] - by_kind['frozen']['token']:+.4f}"
)

# features are matched by identity (their row of active_features), since each graph indexes its own selection
overlap = top_k_overlap(
    influence["frozen"][: len(frozen.selected_features)],
    influence["live"][: len(live.selected_features)],
    ks=(10, 25, 100),
    ids_a=frozen.selected_features.tolist(),
    ids_b=live.selected_features.tolist(),
)
display_notebook_table(
    [[k, f"{v:.3f}"] for k, v in overlap.items()],
    ["k", "Jaccard overlap"],
    title="Do the regimes agree on the most influential features?",
    caption="Overlap of the top-k features by influence, matched by feature identity.",
)
if neumann:
    display_plotly_figure(plot_neumann_convergence(neumann), "it-neumann-regimes")

#### Is the difference more than this prompt?

The nodes of one graph are not independent trials, so no test over this graph's thousands of nodes can say whether
the live regime gives the error nodes less influence in general: they share one forward pass and one set of paths.
The unit that can be repeated is the prompt. The panel asks the same question of one prompt per *state*, each graph
targeting that prompt's capital and city, and counts the prompts where the live regime gives the error nodes less
influence than the frozen one. Under the null that the regime does not move the error sum, each sign is a coin flip,
so the count gets an exact sign test. The prompts share a template, so the result speaks for this template on this
model, not for prompts in general.

In [ ]:
panel_rows, signs = [], []
for city, capital in influence_panel:
    city_prompt = f"Fact: the capital of the state containing {city} is"
    check = check_next_token(model, tokenizer, city, city_prompt, "▁" + capital)
    ids = torch.tensor([tokenizer.encode(w, add_special_tokens=False)[-1] for w in ("▁" + capital, "▁" + city)])
    sums = {}
    for name, freeze in (("frozen", True), ("live", False)):
        graph = it.compute_attribution_graph(
            module, None, None, 0, prompt=city_prompt, attribution_targets=ids, verbose=False, freeze_attention=freeze
        )
        inf = compute_node_influence(graph.adjacency_matrix.float().cpu(), logit_weights(graph))
        sums[name] = influence_by_kind(inf, graph_kinds(graph))
        del graph
    signs.append(sums["live"]["error"] < sums["frozen"]["error"])
    panel_rows.append(
        [
            city,
            capital,
            check.rank,
            f"{sums['frozen']['error']:.4f}",
            f"{sums['live']['error']:.4f}",
            f"{sums['live']['error'] / sums['frozen']['error'] - 1:+.1%}",
            f"{sums['frozen']['error_share']:.1%}",
            f"{sums['live']['error_share']:.1%}",
        ]
    )
display_notebook_table(
    panel_rows,
    ["City", "Capital", "Capital's rank", "Error influence, frozen", "live", "Change", "Error share, frozen", "live"],
    title="Error-node influence under each regime, one prompt per state",
    caption=(
        "Each row compares one prompt's two graphs; the absolute sums scale with that prompt's target probabilities."
    ),
)
n_lower, n_panel = sum(signs), len(signs)
extreme = max(n_lower, n_panel - n_lower)
p_sign = min(1.0, 2 * sum(math.comb(n_panel, k) for k in range(extreme, n_panel + 1)) / 2**n_panel)
print(f"Live error influence lower on {n_lower} of {n_panel} prompts; exact two-sided sign test p = {p_sign:.4f}")

## Head Loadings

`head_loadings` splits the edges into one target across the heads of each requested layer, plus a bypass
share for the part of those edges no head carries. Each number is a share: the head's part of the summed
edges into the target at that layer. That sum is the share's denominator, and it is the thing to look at
before quoting a share: where the summed edges at a layer are small, the same absolute attribution becomes
a large share, and small measurement differences move it a lot. The partition check beside the table is the
internal control: the head shares and the bypass must sum back to the whole, up to the reported partition
error and the bfloat16 rounding of the sum itself (one step, $1/128$, near 1).

The split comes from one backward pass over several copies of the prompt: in copy $h$ a stop-gradient leaves only
head $h$ of the layer live, one copy keeps no head (the bypass), and one keeps every head (the edge). The
`ap_head_loadings_rows` point records those copies, so one source's edge can be rebuilt from its parts below.

In [ ]:
points.reset_trace(points.HEAD_LOADING_ROWS)
loadings = it.head_loadings(module, None, None, 0, prompt=prompt, head_target=answer, head_layers=head_layers)
layers_reported = loadings.head_loading_layers.tolist()
total = float(loadings.head_loading_total)
partitioned = loadings.head_loading_shares.sum(1) + loadings.head_loading_bypass_shares

rows = []
for row, layer in enumerate(layers_reported):
    shares = loadings.head_loading_shares[row]
    top_head = int(shares.abs().argmax())
    rows.append(
        [
            layer,
            top_head,
            f"{float(shares[top_head]):+.3f}",
            f"{float(shares.sum()):+.3f}",
            f"{float(loadings.head_loading_bypass_shares[row]):+.3f}",
            f"{float(partitioned[row]):.4f}",
            f"{float(loadings.head_loading_partition_errors[row]):.1e}",
        ]
    )
order = sorted(range(len(rows)), key=lambda r: -abs(float(rows[r][2])))
display_notebook_table(
    [rows[r] for r in order[:8]],
    ["Layer", "Top head", "Its share", "All heads", "Bypass", "Heads + bypass", "Partition error"],
    title="Where the edges into Austin pass through attention heads (the eight layers with the largest head share)",
    caption=f"Shares of the summed edges into Austin, {total:.4f}; heads + bypass should read 1 at every layer.",
)
assert bool(((partitioned - 1).abs() <= loadings.head_loading_partition_errors + 0.01).all()), (
    "head parts must partition the edges up to the reported partition error and the rounding of their sum"
)
assert float(loadings.head_loading_shares.abs().max()) > 1e-3, "a report of zeros would also partition the edges"

display_plotly_figure(
    plot_head_heatmap(
        loadings.head_loading_shares,
        layers_reported,
        "Each head's share of the summed edges into Austin",
        "share",
        extra_column=("bypass - 1", loadings.head_loading_bypass_shares - 1),
    ),
    "it-head-loadings",
)

The largest head share sits in one layer; the cell below takes that layer's largest source and rebuilds its edge
from the recorded copies. The source is named with the token it fires on and its Neuronpedia description, because a
feature id alone says nothing: in this run it is a layer-20 transcoder feature active on the final ` is`, and the table below gives its
description with a link to the evidence behind it. A feature reading out the unstated *state* at the answer position
is what the two-step question needs, but a description is a hypothesis to check, not a proof: the
intervention section below tests the causal claim directly, without relying on any feature's label.

In [ ]:
if enable_analysis_injection:
    # head-loading rows run over the attribution context's sources: every ACTIVE feature, then errors, then tokens.
    # That layout is not the graph's (which holds only the selected features), so the two must not be mixed up.
    n_active = len(frozen.active_features)
    records = {record["layer"]: record for record in points.read_trace(points.HEAD_LOADING_ROWS)}
    assert sorted(records) == sorted(layers_reported), "one recorded set of copies per reported layer"
    assert all(r["rows"].shape[1] == n_active + (n_layers + 1) * n_pos for r in records.values()), "row layout"
    for row, layer in enumerate(layers_reported):
        rows_l, n_heads = records[layer]["rows"], records[layer]["n_heads"]
        # positive control: the recorded copies reproduce the report's shares
        rebuilt = (rows_l[:n_heads] - rows_l[n_heads]).sum(1) / total
        assert torch.allclose(rebuilt, loadings.head_loading_shares[row].float(), atol=1e-3), f"layer {layer}"
    layer = layers_reported[int(loadings.head_loading_shares.abs().amax(1).argmax())]
    record = records[layer]
    source = int(record["rows"][-1].abs().argmax())
    if source < n_active:
        s_layer, s_pos, s_feat = (int(v) for v in frozen.active_features[source])
        ctx = feature_context([(s_layer, s_pos, s_feat)])
        source_label = ctx.describe(s_layer, s_pos, s_feat)
        display_notebook_table(
            [[ctx.link(s_layer, s_feat), ctx.token(s_pos), ctx.explanation(s_layer, s_feat)]],
            ["Feature", "Fires on", "Neuronpedia description"],
            title=f"Layer {layer}'s largest source into Austin",
            caption="A description is a hypothesis about what the feature responds to; the link shows its evidence.",
        )
    else:
        source_label = f"non-feature source {source - n_active}"
    print(f"Layer {layer} carries the largest head share; its largest source is {source_label}")
    display_plotly_figure(plot_head_split(record, source, source_label, "Austin"), "it-head-split")

## Head Conductance

Conductance answers a different question: not what the graph credits on one prompt, but which heads carry a
*measured change* between two prompts. It integrates each head's pattern (QK) and value (OV) contribution
along the straight path $e(\alpha) = e_0 + \alpha (e_1 - e_0)$ between the Dallas and Detroit embeddings, for the
Austin-versus-Lansing logit difference. Passing the swapped position as `key_positions` also reports the part of
each head's credit that goes through that one key: attention moving toward or away from the city, or the city's
values changing.

**What sums to what.** At each layer, the residual entering the layer (the bypass) and that layer's heads form a
complete cut, so the bypass and the heads' QK and OV parts add up to the measured change at every layer, up to the
completeness residual. Each layer is its own cut: summing heads over layers counts the same change once per layer, and
the bypass is everything that does not pass through that layer's attention (earlier and later attention as well as
every MLP), not an MLP term. So the valid fractions are per layer: the share of the change that passes through a
layer's heads.

**Reading the head table.** Each head's credit is split by mechanism: the pattern (QK) part is its attention weights
moving with its values held, and the value (OV) part is its values changing with its weights held. Each "of which
'via the city'" column is the part of the column to its left that goes through key position 9 alone (the weights on the
city key moving, or the city's value changing), so the column minus it is the credit through every other key. A
query's weights sum to one, so weight moved onto the city is weight moved off the other keys: a QK 'via the city'
larger than its column means the rest has the opposite sign. The change is negative (Austin - Lansing falls), so
negative credit carries it and positive credit opposes it. The last column is the head's QK and OV together as a share
of the whole measured change, which is also the fraction of the change that passes through that head at its layer's
cut. Within a layer, the shares and the bypass add to one; across layers they do not add.

Read `head_conductance_completeness_residual` before any number below it: it is what the integral failed to account
for at each layer, as a fraction of the change. `head_conductance_first_order` is the one-point linearization at the
first prompt; where it disagrees with the measured change, it is the case for integrating along the path at all.

In [ ]:
target_weights = torch.zeros(int(frozen.graph_vocab_size))
target_weights[austin_id], target_weights[lansing_id] = 1.0, -1.0

points.reset_trace(points.CONDUCTANCE_POINTS, points.CONDUCTANCE_RUNNING)
conductance = it.head_conductance(
    module,
    None,
    None,
    0,
    prompt=prompt,
    counterfactual_prompt=swapped_prompt,
    target_weights=target_weights,
    n_steps=conductance_steps,
    key_positions=changed,
)
residual = conductance.head_conductance_completeness_residual.abs()
t_from, t_to = float(conductance.head_conductance_t_from), float(conductance.head_conductance_t_to)
change = t_to - t_from
assert t_from > 0 > t_to, "each prompt's own capital must lead the target on that prompt"
display_notebook_table(
    [
        ["Austin - Lansing on the Dallas prompt", f"{t_from:+.4f}"],
        ["Austin - Lansing on the Detroit prompt", f"{t_to:+.4f}"],
        ["Measured change", f"{change:+.4f}"],
        ["First-order prediction at the Dallas prompt", f"{float(conductance.head_conductance_first_order):+.4f}"],
        [
            f"Largest completeness residual ({conductance_steps} steps)",
            f"{float(residual.max()):.2%} of the change, layer {int(residual.argmax())}",
        ],
    ],
    ["Quantity", "Logits"],
    title="The change being credited",
    caption="Logit differences are the log ratio of the two tokens' probabilities, not a probability.",
)

qk, ov = conductance.head_conductance_qk, conductance.head_conductance_ov
qk_keys, ov_keys = conductance.head_conductance_qk_keys, conductance.head_conductance_ov_keys
head_total = (qk + ov).abs().flatten().topk(8).indices.tolist()
rows = []
for i in head_total:
    layer, head = divmod(i, qk.shape[1])
    rows.append(
        [
            f"L{layer} H{head}",
            f"{float(qk[layer, head]):+.3f}",
            f"{float(qk_keys[layer, head]):+.3f}",
            f"{float(ov[layer, head]):+.3f}",
            f"{float(ov_keys[layer, head]):+.3f}",
            f"{float(qk[layer, head] + ov[layer, head]) / change:+.1%}",
        ]
    )
display_notebook_table(
    rows,
    [
        "Head",
        "Pattern (QK)",
        "of which 'via the city'",
        "Value (OV)",
        "of which 'via the city'",
        "Head's share of the change",
    ],
    title="The heads with the largest conductance",
    caption="Credit in logits of the Austin - Lansing difference; 'via the city' is the part through key position "
    f"{changed[0]} ({prompt_tokens[changed[0]]!r}). A head's share is (QK + OV) / the measured change.",
)
bypass = conductance.head_conductance_resid.float()
heads_by_layer = (qk + ov).sum(1).float()
layer_rows = [
    [
        f"L{layer}",
        f"{float(heads_by_layer[layer]) / change:+.1%}",
        f"{float(bypass[layer]) / change:+.1%}",
        f"{float(conductance.head_conductance_completeness_residual[layer]):+.2%}",
    ]
    for layer in sorted(range(n_layers), key=lambda lyr: -abs(float(heads_by_layer[lyr])))[:6]
]
display_notebook_table(
    layer_rows,
    ["Layer", "Through its heads", "Bypassing them", "Unaccounted (residual)"],
    title="How much of the change passes through each layer's attention (the six largest)",
    caption=(
        "Each row is one complete cut: through + bypassing = 100% + the residual, the integral's overshoot; "
        "rows are different cuts and must not be added."
    ),
)
assert float(qk.abs().max()) > 0 and float(ov.abs().max()) > 0

all_layers = list(range(n_layers))
display_plotly_figure(
    plot_head_heatmap(qk, all_layers, "Pattern (QK) conductance per head (logits)", "credit"), "it-conductance-qk"
)
display_plotly_figure(
    plot_head_heatmap(ov, all_layers, "Value (OV) conductance per head (logits)", "credit"), "it-conductance-ov"
)

### Reading the path

The left panel is the target, Austin minus Lansing in logits, at every point of the path from the Dallas embedding
($\alpha = 0$) to the Detroit one ($\alpha = 1$); the dashed line is what a linearization at the start predicts. The
right panel is its derivative along the path, in logits per unit of $\alpha$: its area is the measured change, and the
conductance of any cut is that area split over the cut's parts. Where the derivative is near zero the swap is doing
nothing yet; where it peaks, the model is switching from the Dallas reading to the Detroit one. A peak far from
$\alpha = 0$ is why the first-order prediction falls short: the slope at the start does not see the switch.

In [ ]:
if enable_analysis_injection:
    path_points = points.read_trace(points.CONDUCTANCE_POINTS)
    # positive control: the recorded path starts and ends at the report's two targets
    assert abs(float(path_points[0]["t"][0]) - t_from) < 1e-3 and abs(float(path_points[-1]["t"][-1]) - t_to) < 1e-3
    assert abs(float(path_points[0]["dt_dalpha"][0]) - float(conductance.head_conductance_first_order)) < 1e-3
    alphas, t_path, slope = conductance_path(path_points)
    half = float(alphas[int(np.argmax(np.abs(t_path - t_from) >= abs(change) / 2))])
    print(
        f"The derivative peaks at alpha = {float(alphas[int(np.abs(slope).argmax())]):.2f} "
        f"({float(slope[int(np.abs(slope).argmax())]):+.1f} logits per unit alpha); "
        f"half the change is done by alpha = {half:.2f}"
    )
    display_plotly_figure(plot_conductance_path(path_points, "Austin - Lansing"), "it-conductance-path")
    display_plotly_figure(
        plot_running_credit(points.read_trace(points.CONDUCTANCE_RUNNING), conductance_steps), "it-conductance-running"
    )

### How many steps the path needs

When the integral's discretization is all the completeness residual measures, it falls as $1/n^2$ in the step
count $n$: doubling the steps divides it by four. The sweep below shows where it stops doing that. Measured
separately with the same integral on a float32 copy of the model, the residual keeps falling fourfold per doubling,
to about $2 \times 10^{-5}$ at 256 steps; in bfloat16, the dtype this notebook runs in, it stops near half a percent
and rises again with more steps. So the floor is the arithmetic, not the path.

The endpoints explain only part of it. bfloat16 keeps 8 significant bits, so near a logit of 8 its spacing is
$2^{-4} = 0.0625$, and rounding the two endpoint logits can move a 15-logit change by up to about 0.05 logits, a few
tenths of a percent. The rest comes from rounding inside every path point's forward and backward pass, which adds up
over the points instead of averaging out; that is why more steps can make the residual worse. Two direct traces of it
appear in this notebook: the endpoint target differs between runs of the same prompt by one bfloat16 step (8.375
against 8.5) depending on the batch it is computed in, and the partition below recomputes the readout in float32 from
the model's own residuals and gets a measured change that differs from the bfloat16 logits' by one to two percent.

In [ ]:
sweep = sorted(set(residual_sweep_steps))
rows, largest = [], []
for steps in sweep:
    report = it.head_conductance(
        module,
        None,
        None,
        0,
        prompt=prompt,
        counterfactual_prompt=swapped_prompt,
        target_weights=target_weights,
        n_steps=steps,
    )
    largest.append(float(report.head_conductance_completeness_residual.abs().max()))
    rows.append(
        [
            steps,
            f"{largest[-1]:.3%}",
            f"{largest[-2] / largest[-1]:.2f}x" if len(largest) > 1 else "-",
            f"{float(report.head_conductance_t_from):+.4f}",
            f"{float(report.head_conductance_t_to):+.4f}",
        ]
    )
display_notebook_table(
    rows,
    ["Steps", "Largest residual (of the change)", "Fall from the previous count", "Target, Dallas", "Target, Detroit"],
    title="The completeness residual against the step count (bfloat16)",
    caption="A pure discretization error falls 4x per doubling of the steps.",
)
assert largest[1] < largest[0], "the residual must fall while the discretization dominates"
display_plotly_figure(plot_residual_convergence(sweep, largest), "it-residual-convergence")

## Who Writes the Change: Attention against MLPs

Conductance credits heads layer by layer; it does not say how the whole change divides between attention and MLPs,
because its cuts overlap. One partition that is exact and disjoint is over the final residual. At the last position the
final residual is the embedding plus every layer's attention output plus every layer's MLP output, so the change of
the logit difference between the two prompts can be split over those terms by integrating the readout's gradient along
the line between the two final residuals. Integrated gradients over a sum is additive over its terms, so the parts add
up to the measured change, with the integral's discretization the only remainder.

**What the split does say.** The swap is at the city's position and the answer is read at the last one, and
positions exchange information only through attention. So every effect of the swap reaches the answer through at least
one attention layer, and an MLP at the last position can only rework what attention has brought there. The split
answers a narrower question than "how much does attention cause": of the change in what the unembedding reads, how much
was last written by an attention layer and how much by an MLP, and at which layers.

What the partition counts is the **direct** path: what each component wrote into the final residual. An effect an
attention layer has by changing what a later MLP computes is credited to that MLP, and an MLP effect that a later
attention head carries to the answer is credited to that head, so "attention writes 70%" is not "70%
of the effect is caused by attention"; the interventions below measure the total effects instead. The units are logits
of the Austin - Lansing difference, the log ratio of the two probabilities: a share of this change is not a share of
probability mass, and there is no partition of a probability change into these components that is both exact and
disjoint, since the softmax couples every logit. The MLP outputs here are the model's own, not the transcoders'
reconstructions: splitting the MLP share between transcoder features and error is what the attribution graph's
feature and error nodes do, for a different target.

In [ ]:
partition = partition_final_residual(module, prompt, swapped_prompt, austin_id, lansing_id)
attn_part, mlp_part, embed_part = partition.total("attn"), partition.total("mlp"), partition.parts["embed"]
display_notebook_table(
    [
        ["Attention outputs (all layers)", f"{attn_part:+.3f}", f"{attn_part / partition.change:.1%}"],
        ["MLP outputs (all layers)", f"{mlp_part:+.3f}", f"{mlp_part / partition.change:.1%}"],
        ["Embedding at the last position", f"{embed_part:+.3f}", f"{embed_part / partition.change:.1%}"],
        [
            "Unaccounted (discretization)",
            f"{partition.unexplained:+.4f}",
            f"{partition.unexplained / partition.change:.2%}",
        ],
        ["Measured change (float32 readout)", f"{partition.change:+.3f}", "100%"],
    ],
    ["Component", "Part of the change (logits)", "Share"],
    title="Direct writes into the final residual: the Dallas-to-Detroit change of Austin - Lansing",
    caption=(
        f"Readout recomputed in float32 from the captured residuals: {partition.t_from:+.3f} -> {partition.t_to:+.3f}, "
        f"against the bfloat16 logits' {t_from:+.3f} -> {t_to:+.3f}. Rebuilding the residual from its terms has "
        f"relative error {partition.recon_error:.1e}."
    ),
)
assert abs(partition.unexplained) < 0.01 * abs(partition.change), "the parts must add up to the measured change"
assert partition.recon_error < 0.02, "the captured terms must sum to the final residual"
display_plotly_figure(
    plot_layer_bars(
        {
            "attention": [partition.parts[f"attn{layer}"] for layer in range(n_layers)],
            "MLP": [partition.parts[f"mlp{layer}"] for layer in range(n_layers)],
        },
        "Each layer's direct write into the change (logits of Austin - Lansing)",
        "part of the change (logits)",
    ),
    "it-residual-partition",
)
top_parts = sorted(partition.parts.items(), key=lambda kv: -abs(kv[1]))[:5]
print("Largest single writers:", ", ".join(f"{name} {value:+.2f}" for name, value in top_parts))

## Changing the Answer: Interventions on Attention and MLP Outputs

Attribution says where credit sits; an intervention says what an edit does. This section edits the natural model,
with its own attention patterns and norms, through `it.model_fwd_intervention`, and reads the answer it gives. Every
edit is at the last position (the `last_token` scope), because that is where the answer is read and where the
partition above located the change. Editing other positions as well needs an operand per position, which the
surface does not take yet: `model_fwd_intervention` edits either the final token or every position with one shared
operand, so an interchange at the city position and the answer position, each with its own delta, cannot be expressed.
venngram's `positions` scope names that edit, and Interpretune refuses a recipe that uses it rather than applying it
to the wrong positions; supporting it is tracked in [#987](https://github.com/speediedan/interpretune/issues/987).

The edits are the Dallas-to-Detroit change of each site's output. Two kinds are run:

- **Interchange** replaces a site's output with its value on the Detroit prompt (`replace`). Replacing every attention
  and every MLP output at the last position rebuilds Detroit's final residual exactly, so that arm must reproduce
  Detroit's answer: it is the positive control for the whole construction.
- **Dose** adds a multiple of the change (`add`), from zero (which must change nothing) upward. Later sites respond to
  an earlier site's edit, so a dose of one is not the interchange: the added change rides on top of the response.

Each arm is compared at the same doses and also against the edit's size, the L2 norm of everything added, in residual
units, so that arms that move more of the residual are not credited for size alone. An arm's size is
$\sqrt{\sum_s \lVert \Delta_s \rVert^2}$ over its sites $s$; the bars after the interchange table show each site's
$\lVert \Delta_s \rVert$. The control for direction is a
random edit of the same size at every site. Sites are grouped as attention only, MLP only and both, over all layers
and over the early and late halves, and each single layer is also edited on its own.

**The arms are addresses.** Each arm is written once, as a venngram site address over a small address book: `@attn`
is `point:hook_attn_out`, `@mlp` is `point:hook_mlp_out`, and `@early` and `@late` are the two halves of the stack
(`layers::13` and `layers:13:` on this 26-layer model). `resolve_sites` turns an address into the concrete sites it
selects on the model at hand, so the late-attention arm is `@attn & @late` rather than a list of 13 hook names. Only
the halves depend on the model's depth, which is what lets the same arms run on another architecture.

In [ ]:
module.analysis_cfg = AnalysisCfg(target_op=it.model_fwd_intervention, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])
_, dallas_terms = capture_residual_terms(module, prompt, n_layers)
_, detroit_terms = capture_residual_terms(module, swapped_prompt, n_layers)
half = n_layers // 2
# the arms are venngram site addresses over a small address book: the attention and MLP writes, and the two halves
arm_sets = intervention_arm_sets(n_layers)
site_sets = resolve_intervention_arms(n_layers, INTERVENTION_ARMS, arm_sets)
display_notebook_table(
    [
        [name, f"`{INTERVENTION_ARMS[name]}`", len(sites), f"{sites[0]} ... {sites[-1]}"]
        for name, sites in site_sets.items()
    ]
    + [[f"set `@{key}`", f"`{value}`", "", ""] for key, value in arm_sets.items()],
    ["Arm", "Address", "Sites", "First ... last site"],
    title=f"The intervention arms as addresses, resolved on this {n_layers}-layer model",
)


def describe(result):
    return [
        f"{result['gap']:+.2f}",
        f"{result['p_a']:.1%}",
        f"{result['p_b']:.1%}",
        f"{result['top'].strip()!r} ({result['top_p']:.0%})",
    ]


rows, interchanges = [], {}
for name, hooks in site_sets.items():
    replacement = {h: detroit_terms[h][0, -1] for h in hooks}
    result = apply_last_token_edits(module, prompt, replacement, 1.0, austin_id, lansing_id, mode="replace")
    interchanges[name] = result
    size = float(torch.sqrt(sum((d**2).sum() for d in last_token_edits(dallas_terms, detroit_terms, hooks).values())))
    rows.append([name, f"`{INTERVENTION_ARMS[name]}`", f"{size:.0f}", *describe(result)])
    if name == "both, all layers":
        interchange_all = result
natural = checks[1]
display_notebook_table(
    rows,
    [
        "Sites replaced with Detroit's",
        "Address",
        "Edit size",
        "Austin - Lansing",
        "p(Austin)",
        "p(Lansing)",
        "Top token",
    ],
    title="Interchange: the Dallas prompt with some last-position outputs taken from the Detroit prompt",
    caption=(
        f"Unedited, the Dallas prompt gives Austin a {checks[0].margin:+.2f}-logit lead; the Detroit prompt's own "
        f"top token is {natural.expected.strip()!r} at {natural.prob:.1%}."
    ),
)
# positive control: replacing every term of the last position's residual reproduces the Detroit prompt's answer
assert interchange_all["top"].strip() == swapped_answer.strip("▁"), interchange_all
assert abs(interchange_all["gap"] - t_to) < 0.25, (interchange_all["gap"], t_to)

# the late-attention interchange alone, token by token: the improbable answer and the two states, before and after
late = interchanges["attention, late half"]
display_topk_token_predictions(
    prompt,
    late["pre_logits"],
    late["post_logits"],
    tokenizer,
    k=5,
    key_tokens=[
        ("Austin", austin_id),
        ("Lansing", lansing_id),
        ("Texas", texas_id),
        ("Michigan", michigan_id),
        ("Dallas", dallas_id),
        ("Detroit", detroit_id),
    ],
)

site_norms = {
    kind: [
        float(last_token_edits(dallas_terms, detroit_terms, [f"blocks.{layer}.hook_{hook}_out"]).popitem()[1].norm())
        for layer in range(n_layers)
    ]
    for kind, hook in (("attention", "attn"), ("MLP", "mlp"))
}
display_plotly_figure(
    plot_layer_bars(
        site_norms, "Size of the Dallas-to-Detroit change at each site (last position)", "L2 norm (residual units)"
    ),
    "it-intervention-site-norms",
)
for kind, norms in site_norms.items():
    early, late_half = math.sqrt(sum(v**2 for v in norms[:half])), math.sqrt(sum(v**2 for v in norms[half:]))
    print(
        f"{kind}: layers 0-{half - 1} {early:.0f}, layers {half}-{n_layers - 1} {late_half:.0f}, all layers "
        f"{math.sqrt(early**2 + late_half**2):.0f} (sizes add in quadrature)"
    )

dose_arms = {
    name: site_sets[name]
    for name in (
        "attention, all layers",
        "MLP, all layers",
        "both, all layers",
        "attention, late half",
        "MLP, late half",
    )
}
dose_results = {}
for name, hooks in dose_arms.items():
    edits = last_token_edits(dallas_terms, detroit_terms, hooks)
    dose_results[name] = [
        apply_last_token_edits(module, prompt, edits, dose, austin_id, lansing_id) for dose in intervention_doses
    ]
zero = [r for rs in dose_results.values() for r in rs if r["dose"] == 0.0]
assert all(abs(r["gap"] - zero[0]["pre_gap"]) < 1e-3 for r in zero), "a zero dose must change nothing"

flip_rows = []
for name, results in dose_results.items():
    flipped = next((r for r in results if r["top"].strip() == swapped_answer.strip("▁")), None)
    crossed = next((r for r in results if r["gap"] < 0), None)
    flip_rows.append(
        [
            name,
            f"{crossed['dose']:.2f} ({crossed['edit_norm']:.0f})" if crossed else "not within the doses",
            f"{flipped['dose']:.2f} ({flipped['edit_norm']:.0f})" if flipped else "never the top token",
            "; ".join(f"{r['dose']:g}: {r['top'].strip()!r}" for r in results),
        ]
    )
display_notebook_table(
    flip_rows,
    [
        "Arm",
        "Lansing passes Austin at dose (size)",
        "Lansing is the top token at dose (size)",
        "Top token at each dose",
    ],
    title="How much of the change each arm needs to change the answer",
    caption="Doses are multiples of the Dallas-to-Detroit change at every site in the arm; size is its L2 norm.",
)
display_plotly_figure(plot_dose_response(dose_results, "Austin", "Lansing"), "it-intervention-doses")

# nonadditivity: the combined arm against the sum of its two halves' changes
for dose in (0.5, 1.0):
    pick = {name: next(r for r in rs if r["dose"] == dose) for name, rs in dose_results.items()}
    base = pick["both, all layers"]["pre_gap"]
    parts = (pick["attention, all layers"]["gap"] - base) + (pick["MLP, all layers"]["gap"] - base)
    print(
        f"Dose {dose}: attention + MLP changes sum to {parts:+.2f}, the combined edit changes the gap by "
        f"{pick['both, all layers']['gap'] - base:+.2f}"
    )

generator = torch.Generator().manual_seed(0)
control_rows = []
for name in ("attention, all layers", "MLP, all layers", "both, all layers"):
    edits = last_token_edits(dallas_terms, detroit_terms, site_sets[name])
    for seed in range(3):
        random_edits = {}
        for h, d in edits.items():
            r = torch.randn(d.shape, generator=generator).to(d.device)
            random_edits[h] = r / r.norm() * d.norm()
        result = apply_last_token_edits(module, prompt, random_edits, 1.0, austin_id, lansing_id)
        control_rows.append([name, seed, f"{result['edit_norm']:.0f}", *describe(result)])
display_notebook_table(
    control_rows,
    ["Arm", "Seed", "Edit size", "Austin - Lansing", "p(Austin)", "p(Lansing)", "Top token"],
    title="Direction control: random edits of the same size at every site (dose 1)",
    caption="If size alone moved the answer, these would; they leave Austin ahead of Lansing.",
)
assert all(float(row[3]) > 0 for row in control_rows), (
    "a random edit of the same size should not flip Austin and Lansing"
)

profile = {"attention": [], "MLP": []}
for layer in range(n_layers):
    for kind, hook in (("attention", f"blocks.{layer}.hook_attn_out"), ("MLP", f"blocks.{layer}.hook_mlp_out")):
        replacement = {hook: detroit_terms[hook][0, -1]}
        result = apply_last_token_edits(module, prompt, replacement, 1.0, austin_id, lansing_id, mode="replace")
        profile[kind].append(result["gap"] - zero[0]["pre_gap"])
display_plotly_figure(
    plot_layer_bars(
        profile, "Change of Austin - Lansing from replacing ONE layer's output with Detroit's", "change (logits)"
    ),
    "it-intervention-layers",
)
top_layers = sorted(((v, k, lyr) for k, vs in profile.items() for lyr, v in enumerate(vs)), key=lambda t: t[0])[:4]
print("Largest single-layer interchanges:", ", ".join(f"{k} L{lyr} {v:+.2f}" for v, k, lyr in top_layers))

## The Same Arms on a Second Model Family

The interchange arms above are addresses, not hook names, so they are not tied to Gemma 2. This section runs them on
`second_model`, Llama 3.2 1B by default: a different family and a different depth (16 layers), loaded as a plain
`(core, nnsight)` composition without transcoders, because an interchange needs only the model and its backend.
Nothing about the arms changes: `resolve_intervention_arms` evaluates the same `INTERVENTION_ARMS` against this
model's own book, in which `@early` and `@late` are `layers::8` and `layers:8:`, and the nnsight backend maps each
`blocks.<layer>.hook_attn_out` and `blocks.<layer>.hook_mlp_out` onto Llama's modules.

The positive control carries over too: replacing every attention and MLP output at the last position must reproduce
the Detroit prompt's answer on this model as well.


In [ ]:
second_module = None
if second_model:
    import dataclasses

    from interpretune.adapters.core import ITModule
    from interpretune.adapters.nnsight.config import ITNNsightConfig, NNsightConfig
    from interpretune.config.mixins import HFFromPretrainedConfig
    from interpretune.config.shared import AutoCompConfig
    from interpretune.harness.sessions import NeutralSessionMixin

    second_tokenizer_kwargs = {"model_input_names": ["input"], "padding_side": "left"}
    second_session = ITSession(
        ITSessionConfig(
            adapter_ctx=(it.Adapter.core, it.Adapter.nnsight),
            datamodule_cfg=dataclasses.replace(
                base_itdm_cfg, model_name_or_path=second_model, tokenizer_kwargs=second_tokenizer_kwargs
            ),
            module_cfg=ITNNsightConfig(
                model_name_or_path=second_model,
                tokenizer_kwargs=second_tokenizer_kwargs,
                os_env_model_auth_key="HF_GATED_PUBLIC_REPO_AUTH_KEY",
                auto_comp_cfg=AutoCompConfig(
                    module_cfg_name="PromptAnalysisConfig", module_cfg_mixin=NeutralSessionMixin
                ),
                hf_from_pretrained_cfg=HFFromPretrainedConfig(
                    pretrained_kwargs={"device_map": "cpu", "dtype": "float32"}
                ),
                nnsight_cfg=NNsightConfig(
                    model_name=second_model,
                    device_map="cuda" if torch.cuda.is_available() else "cpu",
                    torch_dtype="float32",
                    dispatch=True,
                    tokenizer_kwargs={"padding_side": "left"},
                ),
            ),
            datamodule_cls=dm_cls,
            module_cls=ITModule,
        )
    )
    it.it_init(**second_session)
    second_module = second_session.module
    second_module.analysis_cfg = AnalysisCfg(target_op=it.model_fwd_intervention, ignore_manual=True, save_tokens=False)
    init_analysis_cfgs(second_module, [second_module.analysis_cfg])

if second_module is not None:
    second_tokenizer = module_tokenizer(second_module)
    second_layers = int(second_module.model.config.num_hidden_layers)
    word_ids = {
        word: second_tokenizer.encode(f" {word}", add_special_tokens=False)[0]
        for word in ("Austin", "Lansing", "Dallas", "Detroit")
    }
    second_logits, second_dallas = capture_residual_terms(second_module, prompt, second_layers)
    _, second_detroit = capture_residual_terms(second_module, swapped_prompt, second_layers)
    unedited = second_logits.reshape(-1, second_logits.shape[-1])[-1]
    unedited_probs = unedited.softmax(-1)
    rank = int((unedited.argsort(descending=True) == word_ids["Austin"]).nonzero()[0, 0]) + 1
    top_word = second_tokenizer.decode([int(unedited.argmax())])
    p_austin, p_lansing = float(unedited_probs[word_ids["Austin"]]), float(unedited_probs[word_ids["Lansing"]])
    print(
        f"{second_model}: {second_layers} layers. On the Dallas prompt the top token is {top_word!r} "
        f"({float(unedited_probs.max()):.0%}); Austin ranks {rank} ({p_austin:.1%}), Lansing has {p_lansing:.2%}."
    )
    second_sites = resolve_intervention_arms(second_layers)
    second_rows, second_results = [], {}
    for name, hooks in second_sites.items():
        result = apply_last_token_edits(
            second_module,
            prompt,
            {h: second_detroit[h][0, -1] for h in hooks},
            1.0,
            word_ids["Austin"],
            word_ids["Lansing"],
            mode="replace",
        )
        second_results[name] = result
        second_rows.append([name, f"`{INTERVENTION_ARMS[name]}`", len(hooks), *describe(result)])
    display_notebook_table(
        second_rows,
        ["Arm", "Address", "Sites", "Austin - Lansing", "p(Austin)", "p(Lansing)", "Top token"],
        title=f"Interchange on {second_model}: the same arms, resolved on its {second_layers} layers",
        caption=(
            f"Unedited, Austin leads Lansing by {second_results['both, all layers']['pre_gap']:+.2f} logits. Each arm "
            "replaces its sites' last-position outputs with the Detroit prompt's."
        ),
    )
    # positive control on the second model: every attention and MLP output from Detroit gives Detroit's answer
    both = second_results["both, all layers"]
    assert both["top"].strip() == "Lansing", both["top"]
    assert both["gap"] < 0 < both["pre_gap"], (both["gap"], both["pre_gap"])

**Reading the second model.** Llama 3.2 1B answers this prompt less well than Gemma 2: its top token on the Dallas
prompt is ' Dallas' and Austin ranks fifth, so what is compared here is the Austin - Lansing gap rather than the top
answer. The arms say the same thing they said on Gemma 2. Replacing the late half's attention outputs with Detroit's
moves the gap from about +7 logits to about -7 and makes Lansing the top token; the early half's attention leaves the
gap where it was, and the MLP outputs shrink it without making Lansing the top token. The positive control holds on
this model too.

On both models the all-attention arm already gives the full interchange's result. An MLP acts on one position at a
time, so once every attention output at the last position is Detroit's, that position's residual is Detroit's at every
layer, and the MLPs write Detroit's outputs by themselves.

Nothing in the arms was rewritten for Llama: the same addresses resolved on its 16 layers rather than Gemma 2's 26.


## The Unstated Step: Texas, Then Michigan

The prompt never names the *state*, yet the answer depends on it. If the model answers by way of the *state*, the swap
should change a *state* representation before it changes the capital: Texas on the Dallas prompt, Michigan on the
Detroit one. Three readings test that, from the cheapest to the most direct:

- **A logit-lens read of Texas minus Michigan** at every layer, at the city's position and at the answer's. Each
  layer's residual is read through the model's final norm and unembedding, so the number is how strongly the residual
  would favour Texas over Michigan if the model stopped at that layer. A read is a correlate: it shows where the
  information is legible, not that the model uses it there.
- **The features circuit-tracer's own published graph labels "Texas"** for this prompt. Its Neuronpedia demo graph
  <a id="cite-2b"></a>[[2]](#ref-2) groups seven transcoder features into a "Texas" supernode, among them L20 #15589, the
  head-loading section's candidate. They are read on both prompts, beside the features the Detroit prompt activates
  most in their place.
- **A causal test of the read direction.** At each layer, replace only the part of the answer position's residual that
  lies along the read's direction with its Detroit value, and compare with replacing that layer's whole residual. If
  that one direction carries the *state* the answer depends on, moving it alone should move the answer.

In [ ]:
resid = {
    "Dallas": capture_resid_post(module, prompt, n_layers),
    "Detroit": capture_resid_post(module, swapped_prompt, n_layers),
}
state_read = {name: logit_lens_contrast(module, r, texas_id, michigan_id) for name, r in resid.items()}
answer_pos = n_pos - 1
display_plotly_figure(
    plot_state_read(
        state_read,
        {
            city_pos: "at the city (' Dallas' / ' Detroit')",
            answer_pos: f"at the answer position ({prompt_tokens[-1]!r})",
        },
        "Texas - Michigan, read from each layer's residual (logit lens)",
        "Texas - Michigan (logits)",
    ),
    "it-state-read",
)
for pos, where in ((city_pos, "the city"), (answer_pos, "the answer position")):
    gap = (state_read["Dallas"][:, pos] - state_read["Detroit"][:, pos]).cpu()
    peak = int(gap.argmax())
    half = int(torch.nonzero(gap >= 0.5 * float(gap[peak]))[0])
    print(
        f"At {where}: Dallas minus Detroit read {float(gap[0]):+.1f} after layer 0, peaks at {float(gap[peak]):+.1f} "
        f"after layer {peak} (half of the peak first after layer {half}), and ends at {float(gap[-1]):+.1f}"
    )

# (layer, position, feature): the "Texas" supernode of circuit-tracer's published graph for this prompt
texas_supernode = [
    (20, 10, 15589),
    (20, 9, 15589),
    (19, 9, 7477),
    (16, 9, 25),
    (14, 9, 2268),
    (7, 9, 6861),
    (4, 9, 13154),
]
_, dallas_acts = module.replacement_model.get_activations(prompt)
_, detroit_acts = module.replacement_model.get_activations(swapped_prompt)
assert dallas_acts.shape[:2] == (n_layers, n_pos), dallas_acts.shape
shift = (detroit_acts - dallas_acts).float()
shift[:, :city_pos] = 0.0  # positions before the city are identical in both prompts
shift[0] = 0.0  # layer 0 at the city encodes the token itself, not a state
d_tc = shift.shape[-1]
raised = [(i // (n_pos * d_tc), (i // d_tc) % n_pos, i % d_tc) for i in shift.flatten().topk(6).indices.tolist()]
ctx = feature_context(texas_supernode + raised)
on = {city_pos: "' Dallas' / ' Detroit'", answer_pos: repr(prompt_tokens[-1])}
rows = [
    [
        group,
        ctx.link(layer, feat),
        on.get(pos, repr(prompt_tokens[pos])),
        ctx.explanation(layer, feat),
        f"{float(dallas_acts[layer, pos, feat]):.2f}",
        f"{float(detroit_acts[layer, pos, feat]):.2f}",
    ]
    for group, triples in (("labelled Texas", texas_supernode), ("most raised by Detroit", raised))
    for layer, pos, feat in triples
]
display_notebook_table(
    rows,
    ["Group", "Feature", "On", "Neuronpedia description", "Dallas prompt", "Detroit prompt"],
    title="The state features on both prompts (transcoder activations)",
    caption=(
        "'labelled Texas' is the supernode of circuit-tracer's published Neuronpedia graph for this prompt; the "
        "rest are the features whose activation the Detroit prompt raises most, at the city or after it. Descriptions "
        "are hypotheses about what a feature responds to."
    ),
)
active = [(layer, pos, feat) for layer, pos, feat in texas_supernode if float(dallas_acts[layer, pos, feat]) > 0]
kept = [float(detroit_acts[layer, pos, feat] / dallas_acts[layer, pos, feat]) for layer, pos, feat in active]
print(
    f"'Texas' features active on the Dallas prompt: {len(active)} of {len(texas_supernode)}; on the Detroit prompt "
    f"they keep {np.median(kept):.0%} of their Dallas activation (median)"
)

In [ ]:
_, directions = logit_lens_contrast(module, resid["Dallas"][:, answer_pos], texas_id, michigan_id, with_direction=True)
base_gap = zero[0]["pre_gap"]
mediation = []
for layer in range(n_layers):
    hook = f"blocks.{layer}.hook_resid_post"
    r_dallas, r_detroit = resid["Dallas"][layer, answer_pos].float(), resid["Detroit"][layer, answer_pos].float()
    u = directions[layer] / directions[layer].norm()
    whole = apply_last_token_edits(module, prompt, {hook: r_detroit}, 1.0, austin_id, lansing_id, mode="replace")
    along = apply_last_token_edits(
        module, prompt, {hook: torch.dot(r_detroit - r_dallas, u) * u}, 1.0, austin_id, lansing_id
    )
    mediation.append((layer, whole["gap"] - base_gap, along["gap"] - base_gap, along["edit_norm"]))
display_plotly_figure(
    plot_layer_bars(
        {
            "whole residual from Detroit": [m[1] for m in mediation],
            "only along the Texas - Michigan read": [m[2] for m in mediation],
        },
        "Change of Austin - Lansing from replacing the answer position's residual after one layer",
        "change (logits)",
    ),
    "it-state-mediation",
)
display_notebook_table(
    [
        [
            f"L{layer}",
            f"{whole:+.2f}",
            f"{along:+.2f}",
            f"{size:.1f}",
            f"{along / whole:.0%}" if abs(whole) > 0.5 else "-",
        ]
        for layer, whole, along, size in mediation
        if layer % 4 == 0 or layer >= n_layers - 8
    ],
    ["After layer", "Whole residual", "Along the read only", "Size of the one-direction edit", "Share"],
    title="How much of the answer's change one state direction carries",
    caption=(
        "Both edits are at the answer position only. 'Whole residual' replaces it with Detroit's after that layer; "
        "'along the read' changes only its component along the Texas - Michigan read's gradient. Share is shown where "
        "the whole replacement moves the gap by more than half a logit."
    ),
)

**Reading the three results.**

- **The read.** At the city's position the two prompts read as their *states* from layer 0 on: the city tokens' own
  embeddings already point toward their *states*' unembeddings, so at that position the logit lens cannot separate "this
  is Dallas" from "this is in Texas". At the answer position the two prompts read alike through the early and middle
  layers, separate between layers 16 and 18, are furthest apart around layer 22, and come back together in the last two
  layers as the residual turns from the *state* to the capital. That is the timing conductance and the interventions gave
  for the move: late attention, layers 18 and 22.
- **The features.** All seven 'Texas' features fire on the Dallas prompt and are silent on the Detroit one, at the city
  and at the answer position. None of the features the Detroit prompt raises most has a description naming Michigan:
  most read as generic (legal text, place names), one as places around Toledo, Ohio, near Detroit. A description is a
  hypothesis drawn from a feature's top activations, and the 'Texas' feature L20 #15589 is itself described as
  addresses, so this is not evidence against a Michigan representation; finding its features needs the Detroit prompt's
  own graph.
- **The causal test.** Replacing the answer position's whole residual with Detroit's moves the answer from layer 16 on,
  and from layer 22 it moves it almost as far as the full swap: by then the move from the city's position is done.
  Changing only the one direction the Texas - Michigan read is most sensitive to carries about a quarter to a third of
  that change from layer 18 on. So the *state*, as the logit lens reads it, carries part of what the answer depends on,
  and most of the change travels in directions this one read does not see: the read is a partial instrument, and the
  share says how partial.

## The Same Test in a Jacobian-Lens Basis

The logit lens reads a layer's residual as if the model stopped there. A Jacobian lens (J-lens) reads it through the
average effect the rest of the network has on it: the reading row for token $c$ after layer $\ell$ is
$(W_U[c] \odot s) J_\ell$, where $s$ is the final norm's scale and $J_\ell$ is the lens's fitted map for that layer. The
lens used here is the published one for this checkpoint (`neuronpedia/jacobian-lens`, `gemma-2-2b/jlens/`, fitted on
WikiText); nothing in it is fitted on these prompts.

The test is the causal one above, with four directions in place of the read's gradient. Each edit moves only the
answer position's residual along one direction to its Detroit value, after one layer:

- **the logit-lens *state* direction**, Texas minus Michigan's folded unembedding rows;
- **the J-lens *state* direction**, the same rows through $J_\ell$;
- **the J-lens *state* direction with the answer removed**: the J-lens direction minus its projection onto both answer
  directions (Austin minus Lansing, in each lens). What survives cannot move the answer by reading as the answer
  earlier, so if it still moves the answer, the *state* is doing the work;
- **a random direction**, the null.

In [ ]:
from interpretune.analysis.optools import fold_norm_into_unembed_rows, resolve_jlens, resolve_unembed_and_norm_scale

lens = resolve_jlens(module, kind="jlens")
read_rows = (
    fold_norm_into_unembed_rows(
        resolve_unembed_and_norm_scale(module), [texas_id, michigan_id, austin_id, lansing_id], apply_norm=True
    )
    .float()
    .cpu()
)
rand_gen = torch.Generator().manual_seed(0)


def along(v: torch.Tensor, delta: torch.Tensor) -> torch.Tensor:
    # the edit that moves only the residual's component along v to its Detroit value
    u = v / v.norm()
    return torch.dot(delta, u) * u


basis_mediation = []
for layer in [layer for layer in sorted(lens.j_by_layer) if layer >= 14]:
    hook = f"blocks.{layer}.hook_resid_post"
    delta = (resid["Detroit"][layer, answer_pos] - resid["Dallas"][layer, answer_pos]).float().cpu()
    j_rows = read_rows @ lens.j_by_layer[layer].float().cpu()
    answer_dirs = torch.stack([read_rows[2] - read_rows[3], j_rows[2] - j_rows[3]]).double()
    q = torch.linalg.qr(answer_dirs.T)[0]
    jl_state = (j_rows[0] - j_rows[1]).double()
    edits = {
        "logit-lens state": along(read_rows[0] - read_rows[1], delta),
        "J-lens state": along(jl_state.float(), delta),
        "J-lens state, answer removed": along((jl_state - q @ (q.T @ jl_state)).float(), delta),
        "random direction": along(torch.randn(delta.numel(), generator=rand_gen), delta),
    }
    changes = {
        name: apply_last_token_edits(module, prompt, {hook: edit}, 1.0, austin_id, lansing_id)["gap"] - base_gap
        for name, edit in edits.items()
    }
    basis_mediation.append((layer, mediation[layer][1], changes))
display_notebook_table(
    [
        [f"L{layer}", f"{whole:+.2f}"]
        + [
            f"{change:+.2f} ({change / whole:.0%})" if abs(whole) > 0.5 else f"{change:+.2f}"
            for change in changes.values()
        ]
        for layer, whole, changes in basis_mediation
    ],
    ["After layer", "Whole residual", *basis_mediation[0][2]],
    title="The answer's change carried by one direction, by lens basis",
    caption=(
        "Change of Austin - Lansing from moving only the answer position's component along each direction to its "
        "Detroit value; the share of the whole-residual replacement is shown where that moves the gap by more than "
        "half a logit. The J-lens is the published one for this checkpoint, fitted on WikiText."
    ),
)

**Reading the basis comparison.** In short: the J-lens *state* direction moves the answer far more than the logit-lens
one, but mostly because it overlaps the answer direction, so much of what it carries is the capital read early. Only a
small part of it is *state* information that is not the answer.

- **The J-lens *state* direction moves the answer more.** From layer 19 it carries about two thirds of the whole
  replacement's change (67% to 69% at layers 19 to 21, 45% at layer 18), against a quarter to a half for the logit-lens
  *state* direction; the random direction carries nothing.
- **Why: most of that extra effect is the answer, read early.** The J-lens *state* and answer directions are not
  orthogonal (cosine about 0.4), and on this pair the J-lens answer direction alone carries more than the *state*
  direction from layer 20. With both answer directions removed, the J-lens *state* direction keeps most of its size but
  carries only 16% and 22% of the change at layers 18 and 19, and from layer 22 it pushes the gap slightly the other
  way. So a basis that moves the answer more is not, by that alone, reading more of the *state*.
- **What survives is real, and early.** The answer-free part is the *state* information outside both the logit-lens read
  and the answer read. On a panel of non-capital cities (14 under this wording, 182 ordered pairs; 15 under a held-out
  one, 210), it beats a wrong pair of *states* in nearly every pair at layer 19; applied after layer 20 and read at layer
  22, it moves the downstream J-lens *state* read by about 0.45 of its natural difference while moving the capital by only
  0.06 to 0.08 of the full swap. The *state* is carried past the point where the answer is computed, mostly along
  directions aligned with the answer read.
- **The lens matters less than the decomposition.** On the instruction-tuned model, a J++ lens fitted for it (which
  reads held-out items more accurately than the published J-lens) gives the stronger *state* direction at layers 18 and 19
  and the weaker one at layers 21 and 22, while the answer-free component and its dissociation from the answer replicate
  in both. Reading accuracy does not rank bases for intervention at every layer.

The panel, the transfer reads and the J++ comparison are in
[#1062](https://github.com/speediedan/interpretune/issues/1062). Everything here edits the answer position only, so none
of it decomposes the path from the city to the answer: that needs edits at the city's position too, which
[explicit position sets](https://github.com/speediedan/interpretune/blob/main/docs/interpretune_intervention_apis.md#explicit-position-sets)
now express, and replaying the attributed route on the natural model is the follow-up in
[#1014](https://github.com/speediedan/interpretune/issues/1014).

## What Attention-Aware Attribution Adds

Standard circuit-tracer attribution holds attention patterns and norms fixed. Its edges are exact direct effects in
that linearized model, and every effect that works by changing where attention looks is invisible to it. The fork's
additions, available through this surface, report what that leaves out, each with its own units and its own control:

| Surface | What it measured here | What follows |
|----|---------|---------|
| `compute_attribution_graph`, live patterns | Almost all of the disagreement with the frozen graph is on edges into intermediate features, much of it between positions, where only attention can carry credit; the logit rows hold a fraction of a percent of it. | Reading only the answer's row understates the regime's effect: the frozen graph credits the multi-step route through its intermediate features differently. |
| node influence, both regimes | The error nodes take less of the graph's influence under live patterns, on every prompt of the panel, and the token nodes take more. | The reconstruction error is unchanged; what changes is how much pruning-relevant influence the graph cannot assign to an interpretable node. A paired sign test over prompts, not over nodes, is what supports "every prompt". |
| `head_loadings` | The edges into Austin pass through a few late layers' heads, the largest of them reading a feature at the final position, a candidate for the unstated *state* that the interventions test. | Head shares are shares of a stated denominator, with a partition check, not a ranking of heads by importance. |
| `head_conductance` | A measured change is carried by attention at two late layers, about half of it through each layer's heads; each layer's cut is complete on its own. | Valid fractions are per layer; adding layers double-counts. The path's derivative shows the switch happens well inside the path, which a first-order estimate misses. |
| final-residual partition | Of the direct writes into the change, attention outputs account for roughly seven tenths and MLP outputs for three tenths, adding up to the measured change. | An exact split by last writer, in logit units. The swap reaches the answer position only through attention, and effects pass between attention and MLPs in both directions, so it is not a split of total causal effect (or of probability mass). |
| logit lens, *state* features | At the answer position Texas and Michigan separate between layers 16 and 18 and are furthest apart near layer 22; the seven 'Texas' features of circuit-tracer's graph are silent on the Detroit prompt; editing only the read's direction carries about a quarter to a third of the answer's change. | The unstated step is measurable and arrives with the late attention. One read direction carries part of it, so the read is a partial instrument, not the mechanism. |
| `model_fwd_intervention` | Editing attention outputs at the answer position changes the answer to Lansing at half the change; editing MLP outputs moves the logit difference but, across all layers, never makes Lansing the top token, even with a larger edit (the late-layer MLP edit does so at one dose before the answer breaks into fragments); random edits of the same size never make Lansing the top token, though several knock Austin off the top; the early layers do nothing; the combined edit is less than the sum of its parts. | At this position the answer is retrieved through late attention, with late MLPs contributing to the readout; the edits interact, so no additive allocation of the causal effect is implied. |
| the same arms on a second family | On Llama 3.2 1B the arms, unchanged as addresses, give the same picture: late attention alone moves the answer to Lansing, and the MLP arms shrink the gap without changing the answer. | The late-attention finding is not particular to Gemma 2 on this prompt pair, and an arm written as an address carries to another architecture without being rewritten. |

**Across the stages of the question.** The swap changes one token, and the graph's largest differences between regimes
are edges from features on the city's position to features on the final ` is`: the *state* is looked up on the city and
moved to the answer position by attention. The conductance and the interventions locate that movement at a few late
layers' heads, and the strongest feature into the answer at that position reads, on Neuronpedia's description, as the
*state*. The *state*'s own trace agrees: at the answer position Texas and Michigan separate when that late attention acts, and the features circuit-tracer's graph labels "Texas" are silent once the city is Detroit. That is the two-step story the prompt was chosen for; this notebook measures it on one prompt pair, so the
per-component numbers describe this pair, and the panel shows only the influence comparison generalizing across the
template.

The nnsight model-level adapter is the first to support this surface; support in other model-level adapters is
planned. The frozen-regime graph machinery behind the first section is the subject of the
[CT analysis backend demo](ct_analysis_backend_demo.ipynb), and gradient- and ablation-based attribution live in the
[attribution analysis](../attribution_analysis/attribution_analysis.ipynb) notebook.

## References

1. <a id="ref-1"></a>Ameisen, E., Lindsey, J., Pearce, A., et al. "Circuit Tracing: Revealing Computational Graphs in Language Models."
   *Transformer Circuits Thread*, 2025.
   [transformer-circuits.pub/2025/attribution-graphs/methods.html](https://transformer-circuits.pub/2025/attribution-graphs/methods.html)
   [↩](#cite-1)
2. <a id="ref-2"></a>Hanna, M., Piotrowski, M., Lindsey, J., and Ameisen, E. *circuit-tracer*, 2025.
   [github.com/decoderesearch/circuit-tracer](https://github.com/decoderesearch/circuit-tracer). Its Neuronpedia demo
   graph for this prompt, with the "Texas" supernode:
   [gemma-fact-dallas-austin](https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-dallas-austin).
   [↩ a](#cite-2a) [↩ b](#cite-2b)